# Transformers Error Analysis

### 1. Add imports

In [2]:
import pandas as pd 
import torch 

from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
df_test = pd.read_csv("../datasets/clinical_cases_test.csv")

X_test = df_test["medical_abstract"].tolist()

y_test = [
    label - 1 
    for label in df_test["condition_label"].tolist()
]

print(f"Test samples: {len(df_test)}")

Test samples: 2888


In [4]:
MODEL_PATH = "../models/biomedbert-medintake"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

model.to(device)

print(f"Device: {device}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Device: mps


In [5]:
test_encodings = tokenizer(
    X_test,
    trunction=True,
    max_length=256
)

print("Tokenization completed.")

Tokenization completed.


In [6]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [7]:
class MedicalDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item


test_dataset = MedicalDataset(test_encodings, y_test)

print(f"Test samples: {len(test_dataset)}")

Test samples: 2888


In [8]:
test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    collate_fn=data_collator
)

print(f"Batches: {len(test_loader)}")

Batches: 361


In [9]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

print(f"Predictions: {len(all_predictions)}")
print(f"Labels: {len(all_labels)}")


Predictions: 2888
Labels: 2888


In [10]:
errors = df_test.copy()

errors["true_label"] = all_labels
errors["predicted_label"] = all_predictions

errors = errors[
    errors["true_label"] != errors["predicted_label"]
]

print(f"Errors: {len(errors)}")

Errors: 1000


In [13]:
error_pairs = (
    errors
    .groupby(["true_label", "predicted_label"])
    .size()
    .sort_values(ascending=False)
)

print(error_pairs)

true_label  predicted_label
4           3                  199
            0                  161
            1                  136
            2                   99
3           4                   58
2           4                   55
1           0                   44
            4                   37
2           0                   33
            3                   31
0           4                   31
            2                   29
3           2                   18
            0                   14
0           3                   13
            1                   11
2           1                   10
3           1                    9
1           3                    7
            2                    5
dtype: int64
